# Model inversion: a face recovered from a confidence score

Reproduction of **Fredrikson, Jha, Ristenpart (CCS 2015)**,
*"Model Inversion Attacks that Exploit Confidence Information and Basic
Countermeasures"*, dl.acm.org/doi/10.1145/2810103.2813677 -- gradient ascent
on a classifier's own predicted confidence for a target label, starting from
a blank image, recovers a recognizable representative of that class.

Data: **Olivetti/ORL faces** via `sklearn.datasets.fetch_olivetti_faces` -- the
AT&T Laboratories Cambridge face database (40 subjects x 10 images each,
92x112 originally, resampled to 64x64 by scikit-learn), free for research use.
The images depict identifiable real people who consented to research use of
their photos in 1992-94; scikit-learn's docs state only "please give credit
to AT&T Laboratories Cambridge" as a redistribution term -- not a clear grant
to republish an individual photo -- so this notebook and the deck publish
only the per-class **mean** image, referring to subjects by class index only.
See `README.md` for the full licence note.

Model: softmax regression (`nn.Linear`), as in the paper's face experiment.
Deviations from the paper: our objective is `-log p(target|x) + lam *
||x - 0.5||^2`, optimized with Adam and clamped to [0, 1] after every step;
the paper maximizes `1 - p_tilde(target|x)` directly with its own
gradient-descent variant and early stopping, on the original 92x112 ORL
scans rather than this 64x64 resample.


In [ ]:
import json
import os
import random

import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.datasets import fetch_olivetti_faces


## Config

In [ ]:
IMG_SIDE = 64
N_CLASSES = 40
N_PER_CLASS = 10
SAVE_STEPS = [0, 10, 50, 200]
SEED = 0
TARGET = 7      # "person 8"
STEPS = 200
OUT = "out/"
os.makedirs(OUT, exist_ok=True)
torch.manual_seed(SEED)
random.seed(SEED)


## Core: model, inversion loop

The part that matters (kept short so it fits on a slide).

In [ ]:
def make_model():
    return torch.nn.Linear(IMG_SIDE * IMG_SIDE, N_CLASSES)


def train(model, X, y, epochs=2000, lr=0.01, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        opt.zero_grad()
        F.cross_entropy(model(X), y).backward()
        opt.step()
    return model


def invert(model, target, steps, lam=0.015):
    x = torch.full((1, IMG_SIDE * IMG_SIDE), 0.5, requires_grad=True)
    opt = torch.optim.Adam([x], lr=0.05)
    snapshots = {0: x.detach().clone()}
    for step in range(1, steps + 1):
        opt.zero_grad()
        conf = F.log_softmax(model(x), dim=1)[0, target]
        loss = -conf + lam * ((x - 0.5) ** 2).sum()
        loss.backward()
        opt.step()
        with torch.no_grad():
            x.clamp_(0, 1)
        if step in SAVE_STEPS:
            snapshots[step] = x.detach().clone()
    return snapshots


## Data loading and evaluation

Deterministic 7/3 train/test split per class (not scikit-learn's own split,
to keep every one of the 40 classes balanced in both halves).

In [ ]:
@torch.no_grad()
def evaluate(model, X, y):
    return (model(X).argmax(1) == y).float().mean().item()


def load_split(seed):
    data = fetch_olivetti_faces(shuffle=False)
    X, y = torch.tensor(data.data), torch.tensor(data.target, dtype=torch.long)
    train_idx, test_idx = [], []
    for c in range(N_CLASSES):
        idx = list(range(c * N_PER_CLASS, (c + 1) * N_PER_CLASS))
        random.Random(seed + c).shuffle(idx)
        test_idx += idx[:3]
        train_idx += idx[3:]
    return X[train_idx], y[train_idx], X[test_idx], y[test_idx], X, y


def class_mean(X, y, c):
    return X[y == c].mean(dim=0)


def save_png(flat, path, scale=3):
    arr = (flat.reshape(IMG_SIDE, IMG_SIDE).clamp(0, 1).numpy() * 255).round().astype("uint8")
    Image.fromarray(arr, mode="L").resize((IMG_SIDE * scale, IMG_SIDE * scale), Image.NEAREST).save(path)


@torch.no_grad()
def top3(model, flat, labels):
    probs = F.softmax(model(flat.unsqueeze(0))[0], dim=0)
    vals, idx = probs.topk(3)
    return [{"label": labels[i], "p": round(v.item(), 4)} for v, i in zip(vals, idx)]


def corr(a, b):
    return torch.corrcoef(torch.stack([a, b]))[0, 1].item()


def saturated_frac(x):
    return ((x <= 0.001) | (x >= 0.999)).float().mean().item()


## Train the classifier

In [ ]:
name = lambda c: f"person {c + 1}"
labels = [name(c) for c in range(N_CLASSES)]

Xtr, ytr, Xte, yte, Xall, yall = load_split(SEED)
print(f"train={len(Xtr)} test={len(Xte)}")

model = train(make_model(), Xtr, ytr, seed=SEED)
test_acc = evaluate(model, Xte, yte)
print(f"test_acc={test_acc:.4f}")


## Invert the target class, a lam=0 contrast, and a different class

The real optimization dynamic is not "slow convergence to a local maximum":
confidence on the target saturates to p=1.0 by step 5-10 (`||x-0.5||^2`
overshoots to roughly 221 in the process), and the `lam * ||x-0.5||^2`
regularizer then pulls the pixels back toward grey over the following
~100-150 steps, trading confidence (settling at p~0.947) for a cleaner
image. By step 200 this has fully converged -- step 1000 is pixel-identical
to step 200, which is why the script only runs 200 steps.

`lam=0` shows what happens without that pull-back: confidence still
saturates to 1.0, but about half the pixels end up clipped to pure black
or white, and the image correlates far less with the true class mean
(see the printed numbers below).

`other = (target + 1) % 40` -- the slide shows this reconstruction next to
the target's, so it is visible that the two are *different* faces, not an
artifact of the method.

In [ ]:
other = (TARGET + 1) % N_CLASSES
mean_target = class_mean(Xall, yall, TARGET)
mean_other = class_mean(Xall, yall, other)
save_png(mean_target, os.path.join(OUT, "target_mean.png"))
save_png(mean_other, os.path.join(OUT, "other_mean.png"))

snaps = invert(model, TARGET, STEPS)
steps_out = []
for it in SAVE_STEPS:
    file = f"recon_{it:04d}.png"
    save_png(snaps[it][0], os.path.join(OUT, file))
    entry = {"iter": it, "image": file, "top": top3(model, snaps[it][0], labels)}
    if it == STEPS:
        entry["corr_with_mean"] = round(corr(snaps[it][0], mean_target), 4)
    steps_out.append(entry)
    print(it, steps_out[-1]["top"][0])

lam0_snaps = invert(model, TARGET, STEPS, lam=0.0)
lam0_img = lam0_snaps[STEPS][0]
save_png(lam0_img, os.path.join(OUT, "recon_lam0.png"))
print("lam=0:", top3(model, lam0_img, labels)[0],
      "corr_with_mean", round(corr(lam0_img, mean_target), 4),
      "saturated_frac", round(saturated_frac(lam0_img), 4))

other_snaps = invert(model, other, STEPS)
other_file = f"other_{STEPS:04d}.png"
save_png(other_snaps[STEPS][0], os.path.join(OUT, other_file))
print("other", top3(model, other_snaps[STEPS][0], labels)[0])


## Save results.json

In [ ]:
lam0_step = {
    "iter": STEPS, "lam": 0.0, "image": "recon_lam0.png",
    "top": top3(model, lam0_img, labels),
    "corr_with_mean": round(corr(lam0_img, mean_target), 4),
    "saturated_frac": round(saturated_frac(lam0_img), 4),
}
other_step = {"iter": STEPS, "image": other_file, "top": top3(model, other_snaps[STEPS][0], labels)}
results = {
    "dataset": "Olivetti/ORL faces (sklearn.datasets.fetch_olivetti_faces)",
    "size": IMG_SIDE,
    "classes": N_CLASSES,
    "test_acc": round(test_acc, 4),
    "target": TARGET,
    "target_label": name(TARGET),
    "other": other,
    "other_label": name(other),
    "steps": steps_out,
    "lam0_step": lam0_step,
    "other_step": other_step,
}
with open(os.path.join(OUT, "results.json"), "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("done ->", OUT)


## Exercise

1. Vary `lam` in `invert` (the `||x - 0.5||^2` weight, default 0.015). Too
   small (including `lam=0`, run above): confidence still saturates but the
   image comes out noisier and saturated -- about half the pixels pinned to
   pure black or white, and visibly less correlated with the true class
   mean. Too large: the image never leaves grey.
2. Start `x` from random noise (`torch.rand`) instead of a flat 0.5 image
   and compare the step-200 result -- does it still converge to the same
   face, from a different path?
3. The paper's own countermeasure (CCS 2015, Section 6, "Facial
   Recognition"): round the confidence score the model returns to fewer
   decimal places, and re-run `invert` against the rounded model. At what
   rounding level does the attack stop producing a recognizable image? As a
   further extension beyond what the paper tests, try returning only the
   top-1 label with no score at all -- does gradient ascent have anything
   left to climb?
